# Augmentation Pipeline

This notebook standardizes images in `data/face` and augments each person folder for device quality and face angle variation.

Run this notebook first.

In [1]:
from __future__ import annotations

from pathlib import Path
from typing import Optional
from collections import OrderedDict
import random

import cv2
import numpy as np
from numpy.typing import NDArray
from PIL import Image
import face_recognition

ImageArray = NDArray[np.uint8]

try:
    from pillow_heif import register_heif_opener
    register_heif_opener()
    HEIF_ENABLED: bool = True
except Exception:
    HEIF_ENABLED = False

random.seed(42)
np.random.seed(42)

project_root: Path = Path.cwd()
if not (project_root / "data").exists():
    project_root = project_root.parent

FACE_DIR: Path = project_root / "data" / "face"
IMAGE_EXTS: set[str] = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".heic", ".heif"}
AUGMENT_PER_IMAGE: int = 8
JPEG_QUALITY: int = 95
MAX_EDGE: int = 1600

print(f"Face folder: {FACE_DIR}")
print(f"HEIF enabled: {HEIF_ENABLED}")

c:\Users\jaft9\anaconda3\Lib\site-packages\face_recognition_models\__init__.py:7: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_filename


Face folder: c:\Users\jaft9\School\AttSystem\data\face
HEIF enabled: True


In [2]:
raw_dir = Path.cwd().parent / "data" / "face"

if raw_dir is None:
    raise FileNotFoundError(
        "Could not find data/face. Please provide the dataset."
    )

image_exts = {".jpg", ".jpeg", ".png", ".heic"}
counts = OrderedDict()

for folder in sorted([p for p in raw_dir.iterdir() if p.is_dir()]):
    num_images = sum(1 for f in folder.iterdir() if f.is_file() and f.suffix.lower() in image_exts)
    counts[folder.name] = num_images

target = 30

print(f"Dataset folder: {raw_dir}")
print("=" * 40)
for name, count in counts.items():
    remaining = max(0, target - count)
    status = "done" if remaining == 0 else f"need {remaining} more"
    print(f"{name:<15} : {count:>3} images | {status}")

print("=" * 40)
print(f"Total folders: {len(counts)}")
print(f"Total images : {sum(counts.values())}")


Dataset folder: c:\Users\jaft9\School\AttSystem\data\face
benjamin        :  30 images | done
chern_tak       :  30 images | done
chillien        :  22 images | need 8 more
daniel          :  27 images | need 3 more
dylan           :  21 images | need 9 more
han_soon        :  20 images | need 10 more
harry           :  10 images | need 20 more
isaac           :  10 images | need 20 more
jing_ang        :  30 images | done
jun_wei         :  27 images | need 3 more
kang_kai        :  30 images | done
marion          :  28 images | need 2 more
ms_nurul        :  11 images | need 19 more
qi_xuan         :  30 images | done
shuang_quan     :  28 images | need 2 more
wee_xuan        :  24 images | need 6 more
xiang_yue       :  30 images | done
xu_sheng        :  29 images | need 1 more
yoke_hong       :  32 images | done
yong_kang       :  24 images | need 6 more
zi_herng        :  10 images | need 20 more
Total folders: 21
Total images : 503


In [3]:
def load_image(file_path: Path) -> Optional[ImageArray]:
    suffix: str = file_path.suffix.lower()

    if suffix in {".heic", ".heif"}:
        if not HEIF_ENABLED:
            return None
        try:
            with Image.open(file_path) as im:
                rgb = im.convert("RGB")
                arr = np.array(rgb, dtype=np.uint8)
                return cv2.cvtColor(arr, cv2.COLOR_RGB2BGR) # type: ignore
        except Exception:
            return None

    img = cv2.imread(str(file_path), cv2.IMREAD_COLOR)
    if img is None:
        return None
    return img.astype(np.uint8, copy=False)


def resize_if_large(image: ImageArray, max_edge: int = MAX_EDGE) -> ImageArray:
    h, w = image.shape[:2]
    current_max: int = max(h, w)
    if current_max <= max_edge:
        return image
    scale: float = max_edge / float(current_max)
    new_w: int = int(w * scale)
    new_h: int = int(h * scale)
    resized = cv2.resize(image, (new_w, new_h), interpolation=cv2.INTER_AREA)
    return resized.astype(np.uint8, copy=False)


import albumentations as A

# Defines a single augmentation pipeline per the plan
webcam_transform = A.Compose([
    # --- Geometric (both laptop + CCTV) ---
    A.Perspective(scale=(0.05, 0.1), p=0.5),
    A.Rotate(limit=(-15, 15), p=0.4),
    # --- Degradation (CCTV-critical) ---
    A.Downscale(scale_range=(0.25, 0.5), p=0.5),
    A.ImageCompression(quality_range=(30, 70), p=0.5),
    # --- Motion (CCTV walking person) ---
    A.MotionBlur(blur_limit=(5, 9), p=0.4),
    # --- Lighting 
    A.RandomShadow(p=0.6),
    A.RandomBrightnessContrast(brightness_limit=0.5, contrast_limit=0.4, p=0.5),
    A.HueSaturationValue(hue_shift_limit=15, sat_shift_limit=25, val_shift_limit=20, p=0.4),
    A.RandomGamma(gamma_limit=(60, 150), p=0.3),
    # --- Sensor noise ---
    A.GaussNoise(std_range=(0.012, 0.024), p=0.3),
    A.GaussianBlur(blur_limit=(3, 5), p=0.3),
    # --- Occlusion (partial face) ---
    A.CoarseDropout(num_holes_range=(1, 2), hole_height_range=(0.1, 0.15), hole_width_range=(0.1, 0.15), p=0.2)
])

def aug_quality(image: ImageArray) -> ImageArray:
    # A lightweight function for backwards-compatibility or basic variation
    alpha: float = 1.0
    beta: float = 0.0
    return cv2.convertScaleAbs(image, alpha=alpha, beta=beta)

def aug_angle(image: ImageArray) -> ImageArray:
    # Let albumentations handle rotation, we just pass
    return image

def generate_augmented(image: ImageArray, total: int = AUGMENT_PER_IMAGE) -> list[ImageArray]:
    variants = []
    rgb_image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    attempts = 0

    while len(variants) < total and attempts < total * 3:
        augmented = webcam_transform(image=rgb_image)
        aug_bgr = cv2.cvtColor(augmented['image'], cv2.COLOR_RGB2BGR)
        # Guard: only keep if face is still detectable
        locs = face_recognition.face_locations(augmented['image'], model="hog")
        if locs:
            variants.append(aug_bgr)
        attempts += 1

    return variants


In [4]:
def standardize_and_augment_face_folder(face_dir: Path) -> None:
    if not face_dir.exists() or not face_dir.is_dir():
        raise FileNotFoundError(f"Missing dataset folder: {face_dir}")

    person_dirs: list[Path] = sorted([p for p in face_dir.iterdir() if p.is_dir()])
    if not person_dirs:
        raise RuntimeError(f"No people folders found in: {face_dir}")

    total_people: int = 0
    total_original: int = 0
    total_augmented: int = 0

    for person_dir in person_dirs:
        files: list[Path] = sorted([f for f in person_dir.iterdir() if f.is_file() and f.suffix.lower() in IMAGE_EXTS])
        if not files:
            continue

        loaded: list[ImageArray] = []
        for file_path in files:
            img = load_image(file_path)
            if img is None:
                continue
            img = resize_if_large(img)
            loaded.append(img)

        if not loaded:
            continue

        # Reset image files so naming is deterministic every run.
        for old_file in person_dir.iterdir():
            if old_file.is_file() and old_file.suffix.lower() in IMAGE_EXTS:
                old_file.unlink(missing_ok=True)

        person_name: str = person_dir.name
        original_count: int = 0
        aug_count: int = 0

        for idx, img in enumerate(loaded, start=1):
            base_name: str = f"{person_name}_{idx:04d}"
            original_path: Path = person_dir / f"{base_name}.jpg"
            cv2.imwrite(str(original_path), img, [int(cv2.IMWRITE_JPEG_QUALITY), JPEG_QUALITY])
            original_count += 1

            variants: list[ImageArray] = generate_augmented(img, total=AUGMENT_PER_IMAGE)
            for a_idx, var in enumerate(variants, start=1):
                aug_path: Path = person_dir / f"{base_name}_aug_{a_idx:02d}.jpg"
                cv2.imwrite(str(aug_path), var, [int(cv2.IMWRITE_JPEG_QUALITY), JPEG_QUALITY])
                aug_count += 1

        total_people += 1
        total_original += original_count
        total_augmented += aug_count
        print(f"{person_name}: originals={original_count}, augmented={aug_count}, total={original_count + aug_count}")

    print("\nPreprocessing finished")
    print(f"People processed: {total_people}")
    print(f"Standardized originals: {total_original}")
    print(f"Augmented images: {total_augmented}")
    print(f"Grand total in face/: {total_original + total_augmented}")


standardize_and_augment_face_folder(FACE_DIR)

benjamin: originals=30, augmented=196, total=226
chern_tak: originals=30, augmented=240, total=270
chillien: originals=22, augmented=176, total=198
daniel: originals=27, augmented=215, total=242
dylan: originals=21, augmented=164, total=185
han_soon: originals=20, augmented=153, total=173
harry: originals=10, augmented=80, total=90
isaac: originals=10, augmented=40, total=50
jing_ang: originals=30, augmented=216, total=246
jun_wei: originals=27, augmented=216, total=243
kang_kai: originals=30, augmented=240, total=270
marion: originals=28, augmented=224, total=252
ms_nurul: originals=11, augmented=88, total=99
qi_xuan: originals=30, augmented=240, total=270
shuang_quan: originals=28, augmented=213, total=241
wee_xuan: originals=24, augmented=192, total=216
xiang_yue: originals=30, augmented=240, total=270
xu_sheng: originals=29, augmented=232, total=261
yoke_hong: originals=32, augmented=236, total=268
yong_kang: originals=24, augmented=188, total=212
zi_herng: originals=10, augmented=